<center><h1>Last_First_HW3</h1></center>
<br>
<br>

Name: 
<br>
Github Username: 
<br>
USC ID: 

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [15]:
import os                    
import numpy as np             
import pandas as pd           
from scipy.stats import bootstrap     

Get the AReM Data Set

reading the data 


In [ ]:
data_folder = os.path.join("data", "AReM")  

column_names = ["avg_rss12", "var_rss12", "avg_rss13",
                "var_rss13", "avg_rss23", "var_rss23"]   # names for the 6 sensor columns


def read_one_file(file_path):                
    rows = []                                  
    f = open(file_path)                       
    for line in f:                           
        line = line.strip()                   
        if line == "" or line.startswith("#"):   #check if line is empty or not

            continue                           
        line = line.replace(",", " ")              #cleanup the data
        pieces = line.split()                  
        numbers = []                           
        for piece in pieces:                   
            numbers.append(float(piece))      
        rows.append(numbers[1:7])              
    f.close()                                  
    table = pd.DataFrame(rows, columns=column_names)  
    return table                             


activity_folders = ["bending1", "bending2", "cycling", "lying",
                    "sitting", "standing", "walking"]  

total = 0                                               

for activity in activity_folders:                       
    folder_path = os.path.join(data_folder, activity)   
    files = os.listdir(folder_path)                   
    print(activity, len(files))                        
    total = total + len(files)                          

print("Total instances:", total)                       

bending1 7
bending2 6
cycling 15
lying 15
sitting 15
standing 15
walking 15
Total instances: 88


### (b) Test and Train Data

For this section we are separating the trainging and testing data , the test data being bedning 1 and 2, and for every other activity it's data sets 1, 2 ,and 3

In [ ]:
train_list = []                                        # will hold the training instances
test_list = []                                         # will hold the test instances

for activity in activity_folders:                    
    folder_path = os.path.join(data_folder, activity)  
    number_of_files = len(os.listdir(folder_path))     

    if activity == "bending1" or activity == "bending2":   # ppicking the data sets from bending 1 and 2( 1 and 2) and the rest  (1,2,and 3) as the tests
        test_numbers = [1, 2]                         
    else:                                              
        test_numbers = [1, 2, 3]                       

    for file_number in range(1, number_of_files + 1):  #reading the data
        file_name = "dataset" + str(file_number) + ".csv"   
        file_path = os.path.join(folder_path, file_name)    
        table = read_one_file(file_path)              

        instance = {"activity": activity,              # dictionary bundling everything about this file
                    "file_number": file_number,
                    "data": table}

        if file_number in test_numbers:                
            test_list.append(instance)                 # yes -> test pile
        else:
            train_list.append(instance)                # no -> training pile

print("Training instances:", len(train_list))          # test
print("Test instances:", len(test_list))               

for instance in test_list:                              
    print(instance["activity"], instance["file_number"])  

Training instances: 69
Test instances: 19
bending1 1
bending1 2
bending2 1
bending2 2
cycling 1
cycling 2
cycling 3
lying 1
lying 2
lying 3
sitting 1
sitting 2
sitting 3
standing 1
standing 2
standing 3
walking 1
walking 2
walking 3


### (c) Feature Extraction

#### i. Research

the types are:
Statistical: Mean, Median, Variance, std dev, Minimum, Maximum, Range, Interquartile range, skew, root mean square, autocorrelation.

#### ii. Extraction

summarizing the data, instead of having a bunch of data points, we now have the statistical datas obtained using the pyhton stat tools
we summarized all 480 data points into their time series sections and now have 42 summary numbers for each of the 88 instances, making it easier to digest the data and interpret things.

In [ ]:
def extract_features(instance_list):                 
    all_rows = []                                    
    for instance in instance_list:                    
        summary = instance["data"].describe()     
            
        row = {}                                      
        row["activity"] = instance["activity"]        
        for i in range(6):                            
            col = column_names[i]                     
            number = str(i + 1)                       
            row["min" + number] = summary.loc["min", col]        # "min" 
            row["max" + number] = summary.loc["max", col]        # "max"
            row["mean" + number] = summary.loc["mean", col]      # "mean"
            row["median" + number] = summary.loc["50%", col]     # median
            row["std" + number] = summary.loc["std", col]        # std dev
            row["1st_quart" + number] = summary.loc["25%", col]  # 1st quartile
            row["3rd_quart" + number] = summary.loc["75%", col]  # 3rd quartile
        all_rows.append(row)                         
    return pd.DataFrame(all_rows)              

train_features = extract_features(train_list)    
test_features = extract_features(test_list)      


# Test
all_features = pd.concat([train_features, test_features], ignore_index=True) 
all_features.index = all_features.index + 1      

last_instance = test_list[-1]                            
print(last_instance["activity"], last_instance["file_number"])  

print(last_instance["data"]["var_rss23"].describe())       

print(all_features.loc[88, ["min6", "max6", "mean6", "median6",
                            "std6", "1st_quart6", "3rd_quart6"]])   




walking 3
count    480.000000
mean       3.244396
std        1.630983
min        0.000000
25%        2.120000
50%        3.000000
75%        4.240000
max        8.990000
Name: var_rss23, dtype: float64
min6          0.000000
max6          8.990000
mean6         3.244396
median6       3.000000
std6          1.630983
1st_quart6    2.120000
3rd_quart6    4.240000
Name: 88, dtype: float64


#### iii. Standard Deviation

Find the 90% confidence interval using the bootstrap method, by using the python bootstrap tool, we found the standard deviation of the statistic datas that we measured just now.
Because we only have 88 values to sample from, using the bootstrap tool we can resample multiple times, in this case 1000 times and see where the 90% interval of confidence is with those data points.
Thus after seeing the results of those we are then able to see where they lie and be able to identify the ones that are fairly different from each other which will allow us to make predictions on future data.

In [ ]:

def sample_std(x):                                # statistic to compute on each resample
    return np.std(x, ddof=1)                     

feature_names = all_features.columns[1:]          #skip "activity"
results = []                                      

for feature in feature_names:                     
    values = all_features[feature].values       
    original_std = sample_std(values)             
    result = bootstrap((values,),sample_std,confidence_level=0.90,n_resamples=1000, method="percentile", random_state=67)          
    results.append({"feature": feature,"std": original_std,"CI_lower": result.confidence_interval.low,"CI_upper": result.confidence_interval.high})

ci_lookup = ci_table.set_index("feature")              # row labels = "min1", "max1", ...

feature_types = ["min", "max", "mean", "median", "std", "1st_quart", "3rd_quart"]
#used AI to make a easy to read table to interpret the data
readable_rows = []                                     # one row per feature type
for feature_type in feature_types:                     # "min", "max", ...
    row = {"feature": feature_type}                    # first column = the feature type
    for s in range(1, 7):                              # series 1 to 6
        name = feature_type + str(s)                   # e.g. "mean3"
        std_value = ci_lookup.loc[name, "std"]         # the std
        low = ci_lookup.loc[name, "CI_lower"]          # lower end of the 90% CI
        high = ci_lookup.loc[name, "CI_upper"]         # upper end of the 90% CI
        row["series " + str(s)] = f"{std_value:.2f} [{low:.2f}, {high:.2f}]"   # "5.34 [4.70, 5.89]"
    readable_rows.append(row)                          # save this row

readable_table = pd.DataFrame(readable_rows)           # list of dicts -> table
print(readable_table.to_string(index=False))           # print without row numbers

  feature           series 1          series 2          series 3          series 4          series 5          series 6
      min 9.57 [8.27, 10.79] 0.00 [0.00, 0.00] 2.96 [2.75, 3.10] 0.00 [0.00, 0.00] 6.12 [4.36, 7.50] 0.05 [0.00, 0.08]
      max  4.39 [3.30, 5.26] 5.06 [4.63, 5.40] 4.88 [4.13, 5.50] 2.18 [1.97, 2.36] 5.74 [4.75, 6.61] 2.52 [2.24, 2.75]
     mean  5.34 [4.69, 5.86] 1.57 [1.40, 1.70] 4.01 [3.42, 4.49] 1.17 [1.08, 1.22] 5.68 [4.42, 6.71] 1.15 [1.06, 1.21]
   median  5.44 [4.76, 6.02] 1.41 [1.24, 1.54] 4.04 [3.41, 4.54] 1.15 [1.06, 1.20] 5.81 [4.50, 6.90] 1.09 [1.00, 1.15]
      std  1.77 [1.57, 1.95] 0.88 [0.80, 0.94] 0.95 [0.77, 1.13] 0.46 [0.42, 0.48] 1.02 [0.82, 1.23] 0.52 [0.48, 0.55]
1st_quart  6.15 [5.55, 6.64] 0.95 [0.83, 1.03] 4.22 [3.63, 4.69] 0.84 [0.78, 0.89] 6.10 [4.80, 7.19] 0.76 [0.69, 0.81]
3rd_quart  5.14 [4.31, 5.83] 2.13 [1.89, 2.29] 4.17 [3.53, 4.69] 1.55 [1.44, 1.63] 5.53 [4.35, 6.50] 1.52 [1.41, 1.60]


#### iv. Select Features

based on the table, min doesn't seem to variate much between data points and thus doesn't tell us much to differentiate the activites,
meanwhile, max, mean, meadian, and std seem to have more variation between activities, which will make it more useful to make guesses based on them since they differentiate from each other.

in this case we might even drop median or mean because they are very similar to each other therefore I believe the most useful time-domain features are: mean, max, and std

## 2. ISLR 3.7.4

### (a) Linear Train

a) we would expect the cubic to be lower, because the linear model is included in the cubic model, when beta2 and beta3 are equal to 0.

### (b) Linear Test

b) in this case, the linear will have a lower RSS, the linear matches the data since it is trained on it, the cubic will be more flexible and react more to noise from the training data, which is not present in the test.

### (c) Not Linear Train

c) similar to part a, becasue the linear is included in the cubic model, the cubic model will at worst be the same as the linear, it has the advantage that it will bend more towards the real curve than the linear model.

### (d) Not Linear Testing

d) in this case we don't have enough information to tell, since if the relationship is close to linear then the linear RSS will be closer, while if the test is far from linear the cubic will shape better to it.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 